# Image Dataset and DataLoader Setup

This notebook contains the necessary imports, data downloads, transforms, and DataLoaders copied from `pytorch_custom_dataset.ipynb` to prepare the image data for a new model.

In [5]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import requests
import zipfile
from pathlib import Path
import os

# Device-agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [6]:
# Setup path to a datafolder
data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi"

# If the image folder does not exist, download and prepare it
if image_path.is_dir():
    print(f"{image_path} Directory already exists... skipping download")
else:
    print(f"{image_path} does not exist creating one")
    image_path.mkdir(parents=True, exist_ok=True)
    
    # Download the dataset
    zip_filepath = data_path / "pizza_steak_sushi.zip"
    with open(zip_filepath, "wb") as f:
        request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/refs/heads/main/data/pizza_steak_sushi.zip")
        print("Downloading pizza steak sushi data...")
        f.write(request.content)
        
    # Unzip pizza steak sushi file
    with zipfile.ZipFile(zip_filepath, "r") as zip_ref:
        print("Unzipping pizza steak sushi data...")
        zip_ref.extractall(image_path)

data\pizza_steak_sushi Directory already exists... skipping download


In [7]:
# Setup train and test paths
train_dir = image_path / "train"
test_dir = image_path / "test"
train_dir, test_dir

(WindowsPath('data/pizza_steak_sushi/train'),
 WindowsPath('data/pizza_steak_sushi/test'))

In [8]:
# Write a transform for the images
data_transform = transforms.Compose([
    # Resize our images to 64 * 64
    transforms.Resize(size=(64, 64)),
    # Flip the images randomly on the horizontal
    transforms.RandomHorizontalFlip(p=0.5),
    # Convert to Tensor
    transforms.ToTensor()
])

In [9]:
# Turn train and test datasets into ImageFolder datasets
train_data = datasets.ImageFolder(root=train_dir, transform=data_transform, target_transform=None)
test_data = datasets.ImageFolder(root=test_dir, transform=data_transform, target_transform=None)

class_names = train_data.classes
class_dict = train_data.class_to_idx

print(f"Train data length: {len(train_data)}")
print(f"Test data length: {len(test_data)}")
print(f"Class Names: {class_names}")

Train data length: 225
Test data length: 75
Class Names: ['pizza', 'steak', 'sushi']


In [10]:
# Turn datasets into DataLoaders
BATCH_SIZE = 32
NUM_WORKERS = 0  # 0 is recommended for compatibility on Windows

train_dataloader = DataLoader(
    dataset=train_data,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=True
)

test_dataloader = DataLoader(
    dataset=test_data,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=False
)

print(f"Train Dataloader batches: {len(train_dataloader)}")
print(f"Test Dataloader batches: {len(test_dataloader)}")

Train Dataloader batches: 8
Test Dataloader batches: 3


In [11]:
# Retrieve a single batch to verify shapes
img_batch, label_batch = next(iter(train_dataloader))
print(f"Image batch shape: {img_batch.shape} -> [batch_size, color_channels, height, width]")
print(f"Label batch shape: {label_batch.shape}")

Image batch shape: torch.Size([32, 3, 64, 64]) -> [batch_size, color_channels, height, width]
Label batch shape: torch.Size([32])


### 7.2 Create a TinyVGG model class


In [12]:
class TinyVGG(nn.Module):
    """
        Model architecture copying TinyVgg architecture from CNN explainer
    """
    def __init__(self,input_shape:int,hidden_units:int,output_shape:int) -> None:
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape,out_channels=hidden_units,
            kernel_size=3,stride= 1 , padding= 1 ),
            
            nn.ReLU(),
            
            nn.Conv2d(in_channels=hidden_units,out_channels=hidden_units,
            kernel_size=3, stride=1 ,padding=1   ),

            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2)
        )
            

        self.conv_block_2 = nn.Sequential(
        nn.Conv2d(in_channels=input_shape,out_channels=hidden_units,
        kernel_size=3,stride= 1 , padding= 1 ),
        
        nn.ReLU(),
        
        nn.Conv2d(in_channels=hidden_units,out_channels=hidden_units,
        kernel_size=3, stride=1 ,padding=1   ),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=hidden_units,out_features=output_shape    )
        )
    def forward(self,x):
        x = self.conv_block_1(x)
        print(x.shape)
        x = self.conv_block_2(x)
        print(x.shape)
        x = self.classifier(x)
        print(x.shape)
        return x
        #return self.classifier(self.conv_block_2(self.conv_block_1(x)))
        #https://horace.io/brrr_intro.html

In [14]:
torch.manual_seed(42)
model_0 = TinyVGG(input_shape=3,hidden_units=10,output_shape= len(class_names)).to(device)

model_0


TinyVGG(
  (conv_block_1): Sequential(
    (0): Conv2d(3, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block_2): Sequential(
    (0): Conv2d(3, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=10, out_features=3, bias=True)
  )
)